# Laboratorio 02 â€” Notebook GenÃ©rico de IngestiÃ³n Bronze

**Semana:** 04 | **Actividad de referencia:** Actividad 02  
**Modalidad:** Individual | **Entorno:** Databricks (Unity Catalog)

---

## Instrucciones generales

DiseÃ±a un notebook de ingestiÃ³n Bronze **genÃ©rico y reutilizable** que acepte widgets para el formato de entrada, la ruta del archivo, el nombre de la tabla destino y el modo de escritura. El notebook debe poder ingestar tu dataset propio en Bronze **sin cambiar el cÃ³digo**, solo cambiando los widgets.

## Parte 1 â€” DescripciÃ³n del dataset y diseÃ±o del notebook

1. **Nombre, fuente y URL** del dataset.
2. **Formato del archivo:** CSV, JSON, Parquet u otro. Â¿Por quÃ© ese formato?
3. **Esquema esperado:** Lista las columnas y sus tipos. Â¿UsarÃ¡s `inferSchema` o definirÃ¡s el esquema explÃ­cito? Â¿Por quÃ©?
4. **DiseÃ±o de la interfaz de widgets:** Â¿QuÃ© parÃ¡metros deberÃ­a recibir el notebook para ser verdaderamente genÃ©rico?

**Escribe tu respuesta aquÃ­:**

## Parte 2 â€” DefiniciÃ³n de Widgets

In [ ]:
dbutils.widgets.removeAll()

# Formato de entrada
dbutils.widgets.dropdown(
    name         = "formato",
    defaultValue = "csv",
    choices      = ["csv", "json", "parquet", "delta"],
    label        = "Formato del archivo fuente"
)

# Ruta del archivo en el volumen
dbutils.widgets.text(
    name         = "ruta_origen",
    defaultValue = "/Volumes/workspace/default/week_4/tu_archivo.csv",
    label        = "Ruta completa al archivo fuente"
)

# Nombre de la tabla Delta destino
dbutils.widgets.text(
    name         = "tabla_destino",
    defaultValue = "workspace.default.bronze_mi_dataset",
    label        = "Tabla Delta destino (catalog.schema.tabla)"
)

# Modo de escritura
dbutils.widgets.dropdown(
    name         = "modo_escritura",
    defaultValue = "overwrite",
    choices      = ["overwrite", "append", "ignore", "errorifexists"],
    label        = "Modo de escritura Delta"
)

# Inferir esquema automÃ¡ticamente
dbutils.widgets.dropdown(
    name         = "inferir_schema",
    defaultValue = "true",
    choices      = ["true", "false"],
    label        = "Inferir esquema automÃ¡ticamente"
)

print("âœ“ Widgets definidos")

## Parte 3 â€” Leer parÃ¡metros y preparar la lectura

In [ ]:
formato       = dbutils.widgets.get("formato")
ruta_origen   = dbutils.widgets.get("ruta_origen")
tabla_destino = dbutils.widgets.get("tabla_destino")
modo          = dbutils.widgets.get("modo_escritura")
inferir       = dbutils.widgets.get("inferir_schema") == "true"

print(f"  formato       = {formato}")
print(f"  ruta_origen   = {ruta_origen}")
print(f"  tabla_destino = {tabla_destino}")
print(f"  modo          = {modo}")
print(f"  inferir       = {inferir}")

## Parte 4 â€” IngestiÃ³n genÃ©rica multiformato

In [ ]:
# Opciones de lectura por formato
opciones_por_formato = {
    "csv":     {"header": "true",  "inferSchema": str(inferir).lower()},
    "json":    {"multiLine": "true"},
    "parquet": {},
    "delta":   {}
}

reader = spark.read.format(formato)
for k, v in opciones_por_formato.get(formato, {}).items():
    reader = reader.option(k, v)

df_bronze = reader.load(ruta_origen)
print(f"âœ“ Lectura exitosa: {df_bronze.count():,} filas | {len(df_bronze.columns)} columnas")
df_bronze.printSchema()

## Parte 5 â€” Perfil del dataset antes de escribir (quality gate)

In [ ]:
from pyspark.sql import functions as F

total = df_bronze.count()

# Porcentaje de nulos por columna
nulos = df_bronze.select([
    F.round(
        F.sum(F.when(F.col(c).isNull() | (F.col(c).cast("string") == ""), 1).otherwise(0))
        * 100.0 / total, 1
    ).alias(f"{c}_pct_nulos")
    for c in df_bronze.columns
])
nulos.show(truncate=False)

In [ ]:
# EstadÃ­sticas descriptivas
df_bronze.describe().show(truncate=False)

**Calidad del dato antes de Bronze:**  
Â¿Hay columnas con >30% de nulos que deberÃ­as documentar? Â¿Encontraste tipos de dato incorrectos (ej. nÃºmero como string)?  
Documenta aquÃ­ los hallazgos para que queden en el historial del notebook.

## Parte 6 â€” AÃ±adir metadatos de auditorÃ­a y escribir en Bronze

In [ ]:
from datetime import datetime

# AÃ±adir columnas de auditorÃ­a (patrÃ³n estÃ¡ndar de capa Bronze)
df_con_meta = df_bronze \
    .withColumn("_ingest_timestamp", F.current_timestamp()) \
    .withColumn("_source_file",      F.lit(ruta_origen)) \
    .withColumn("_source_format",    F.lit(formato))

# Escribir como tabla Delta
df_con_meta.write \
    .format("delta") \
    .mode(modo) \
    .saveAsTable(tabla_destino)

print(f"âœ“ Tabla Bronze creada: {tabla_destino}")
print(f"  Filas escritas: {df_con_meta.count():,}")
print(f"  Modo:           {modo}")

In [ ]:
# Verificar el historial de la tabla Delta creada
spark.sql(f"DESCRIBE HISTORY {tabla_destino}").show(5, truncate=False)

## Parte 7 â€” Probar el notebook con un segundo dataset

Cambia los widgets (sin modificar el cÃ³digo) para apuntar a un segundo archivo diferente al original, con modo `append` o `overwrite`. Luego ejecuta el notebook desde la Parte 3 y verifica el resultado.

In [ ]:
# Verificar que la tabla Delta contiene los datos del segundo dataset (o el acumulado si usaste append)
spark.table(tabla_destino).groupBy("_source_file").count().orderBy("_source_file").show(truncate=False)

**Preguntas de negocio:**
1. Â¿CuÃ¡ntas filas provienen de cada archivo fuente?
2. Â¿El modo `append` duplicÃ³ algÃºn registro o funcionÃ³ correctamente?
3. Â¿QuÃ© ventaja tiene agregar `_ingest_timestamp` y `_source_file` en Bronze?

In [ ]:
# Exportar el resultado como indicador para el notebook padre
filas_escritas = spark.table(tabla_destino).count()
dbutils.notebook.exit(str(filas_escritas))

## Parte 8 â€” ReflexiÃ³n final

1. Â¿QuÃ© diferencia hay entre `inferSchema=True` y definir el esquema explÃ­citamente? Â¿CuÃ¡l es mÃ¡s seguro en producciÃ³n y por quÃ©?
2. Â¿QuÃ© sucede si usas modo `errorifexists` y la tabla ya existe? Â¿CÃ³mo lo manejarÃ­as en un pipeline programado?
3. Â¿Por quÃ© la capa Bronze deberÃ­a guardar los datos sin transformaciones y con columnas de auditorÃ­a?
4. Â¿CÃ³mo extenderÃ­as este notebook para soportar tambiÃ©n archivos en S3 o Azure Data Lake?

---

## Entrega en Git

```bash
# Copia el template a tu carpeta (solo la primera vez)
# cp semana_04/laboratorios/lab_02_bronze_generico.ipynb semana_04/laboratorios/<tu-nombre>/lab_02_bronze_generico.ipynb

git add semana_04/laboratorios/<tu-nombre>/lab_02_bronze_generico.ipynb
git commit -m "lab: semana04 lab02 bronze generico multiformato <nombre-dataset> - <tu-nombre>"
git push origin develop
```